# 💡 07 · Fluorescence Microscopy Instrumentation
### Catching a single color of light out of a million-fold brighter background

**Part B · Seeing like a microscope** · Module 07 of 37 · *BiologyToBioimage*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/07_fluorescence_microscopy_instrumentation.ipynb)

| | |
|---|---|
| **Hardware target** | CPU only (no GPU needed) |
| **Core frameworks** | NumPy, Matplotlib |
| **Prerequisites** | Modules 03–06 |
| **Estimated time** | 70–85 minutes |
| **Instrumentation topics covered** | The Jablonski diagram and the Stokes shift; fluorophore excitation/emission spectra; the epifluorescence filter cube (excitation filter, dichroic mirror, emission filter); spectral bleed-through between fluorophores; choosing compatible fluorophore/filter combinations for multi-color imaging |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Every spectrum, filter curve and bleed-through calculation here is a parametric Gaussian/logistic MODEL of published peak wavelengths, not a scanned spectrum read from a manufacturer's file.

## 🎯 Learning objectives

1. Explain the Jablonski diagram's basic story — absorption, fast vibrational relaxation, then
   emission — and why it necessarily produces a Stokes shift: emitted light always at a longer
   wavelength than absorbed light.
2. Simulate realistic excitation and emission spectra for a fluorophore, and compute its Stokes
   shift directly from the two peaks.
3. Explain how a filter cube's three components — excitation filter, dichroic mirror, emission
   filter — work together to deliver excitation light to the specimen and only emitted light to the
   detector, and simulate a dichroic's transmission curve.
4. Simulate and quantify spectral bleed-through (crosstalk) between two fluorophores' emission
   spectra through a given filter set, and explain why it corrupts multi-channel quantitative
   measurements such as colocalization.
5. Choose a compatible pair of fluorophores and filter sets for a two-color experiment, and justify
   the choice using a simulated bleed-through comparison.

## 🗺️ Contents

1. The Jablonski diagram and the Stokes shift
2. Excitation and emission spectra: a fluorophore's optical fingerprint
3. The filter cube: excitation filter, dichroic mirror, emission filter
4. Spectral bleed-through: when two fluorophores' colors overlap
5. Choosing fluorophores and filters for multi-color imaging

## 🧫 The biological question

A cell biologist wants to image a green fluorescent reporter and a red fluorescent reporter in the
same live cell at the same time, to test whether the two proteins they label physically colocalize.
If even 15% of the green fluorophore's emitted light leaks through into the channel meant to capture
only the red fluorophore, every colocalization measurement from that experiment is contaminated
before a single line of image-analysis code runs — and the contamination looks exactly like real
biology, not like an error message. Picking fluorophores and filters that keep this leakage small is
an instrumentation decision made weeks before the experiment, not something any downstream algorithm
can fully undo afterward.

---
## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
print('No GPU is needed anywhere in this notebook — everything is plain NumPy/Matplotlib.')

In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
SEED = 7
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11

### 🧰 Course toolkit

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.1; copied so this notebook runs alone) ──
import numpy as np

def fluorophore_spectrum(wavelengths_nm: np.ndarray, peak_nm: float, fwhm_nm: float) -> np.ndarray:
    """A Gaussian-shaped excitation or emission spectrum, normalised to a peak of 1.0, from a
    published peak wavelength and full width at half maximum (both in nm) — a parametric model,
    not a scanned or stored spectrum."""
    sigma = fwhm_nm / (2 * np.sqrt(2 * np.log(2)))
    return np.exp(-((wavelengths_nm - peak_nm) ** 2) / (2 * sigma ** 2))


def dichroic_transmission(wavelengths_nm: np.ndarray, cutoff_nm: float, steepness_nm: float = 5.0) -> np.ndarray:
    """A long-pass dichroic/emission-filter transmission curve: ~0 below ``cutoff_nm``, ~1 above,
    with a logistic transition of width ``steepness_nm`` (a simplified filter model)."""
    return 1.0 / (1.0 + np.exp(-(wavelengths_nm - cutoff_nm) / steepness_nm))

---
## 1 · The Jablonski diagram and the Stokes shift

A fluorophore absorbs a photon and jumps from its ground electronic state ($S_0$) to an excited
state ($S_1$), typically landing in one of several closely-spaced vibrational sub-levels above $S_1$'s
lowest energy. Before it can emit a photon, it rapidly loses a little energy as heat, relaxing down to
the lowest vibrational level of $S_1$ — a process far faster than emission itself. Only *then* does it
emit a photon, falling back to $S_0$. Because some energy was already lost as heat during that
relaxation step, **the emitted photon always carries less energy — and therefore a longer
wavelength — than the absorbed photon.** This energy gap between excitation and emission peaks is the
**Stokes shift**, and it is not a property of any particular fluorophore's chemistry so much as an
unavoidable consequence of this absorb-relax-emit sequence.

In [ ]:
# ── 1.1 a schematic Jablonski diagram, drawn entirely with code ──
fig, ax = plt.subplots(figsize=(6, 4.5))
s0_y, s1_y = 0.0, 1.0
ax.hlines(s0_y, 0.1, 0.9, color='black', lw=2)
ax.hlines(s1_y, 0.1, 0.9, color='black', lw=2)
for dy in (0.06, 0.12):  # a couple of vibrational sublevels above S1's lowest level
    ax.hlines(s1_y + dy, 0.1, 0.9, color='gray', lw=1)
ax.text(0.92, s0_y, '$S_0$ (ground state)', va='center', fontsize=10)
ax.text(0.92, s1_y + 0.12, '$S_1$ (excited state)', va='center', fontsize=10)

ax.annotate('', xy=(0.3, s1_y + 0.12), xytext=(0.3, s0_y),
            arrowprops=dict(arrowstyle='->', color='blue', lw=2))
ax.text(0.22, 0.55, 'absorption\n(short wavelength,\nhigh energy)', color='blue', fontsize=8, ha='right')

ax.annotate('', xy=(0.5, s1_y), xytext=(0.5, s1_y + 0.12),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5, linestyle='dashed'))
ax.text(0.53, s1_y + 0.08, 'vibrational\nrelaxation (heat)', color='gray', fontsize=8)

ax.annotate('', xy=(0.7, s0_y), xytext=(0.7, s1_y),
            arrowprops=dict(arrowstyle='->', color='red', lw=2))
ax.text(0.78, 0.5, 'emission\n(longer wavelength,\nlower energy)', color='red', fontsize=8)

ax.set_xlim(0, 1.5); ax.set_ylim(-0.15, 1.3)
ax.axis('off')
ax.set_title('A simplified Jablonski diagram')
plt.tight_layout(); plt.show()

🔍 **What just happened?** The absorption arrow (blue) is shorter than the emission arrow (red)
in energy terms — it spans a bigger vertical jump but starts from the bottom of $S_0$, while emission
starts from the *relaxed* bottom of $S_1$, always landing with strictly less energy than was absorbed.
That single relaxation step, drawn here as the short gray dashed arrow, is the entire physical origin
of the Stokes shift computed quantitatively in Section 2.

---
## 2 · Excitation and emission spectra: a fluorophore's optical fingerprint

In practice, a fluorophore is characterized not by two single wavelengths but by two smooth
curves: an **excitation spectrum** (how efficiently it absorbs light as a function of wavelength) and
an **emission spectrum** (how its emitted light is distributed across wavelength). Both are reasonably
well approximated, for many common fluorophores, by a Gaussian centered on a published peak
wavelength with a published full width at half maximum (FWHM) — exactly what `fluorophore_spectrum`
computes.

In [ ]:
# ── 2.1 a simulated GFP-like fluorophore's excitation and emission spectra ──
wavelengths_nm = np.linspace(400, 750, 700)
gfp_ex = fluorophore_spectrum(wavelengths_nm, peak_nm=488, fwhm_nm=25)
gfp_em = fluorophore_spectrum(wavelengths_nm, peak_nm=509, fwhm_nm=35)

stokes_shift_nm = 509 - 488
print(f'GFP-like fluorophore: excitation peak 488 nm, emission peak 509 nm')
print(f'Stokes shift: {stokes_shift_nm} nm')

fig, ax = plt.subplots(figsize=(8, 3.3))
ax.fill_between(wavelengths_nm, gfp_ex, color='blue', alpha=0.3, label='excitation spectrum')
ax.fill_between(wavelengths_nm, gfp_em, color='green', alpha=0.3, label='emission spectrum')
ax.axvline(488, color='blue', ls='--', lw=1)
ax.axvline(509, color='green', ls='--', lw=1)
ax.set_xlabel('wavelength (nm)'); ax.set_ylabel('normalized intensity')
ax.set_title('A simulated GFP-like fluorophore: excitation and emission spectra')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The excitation (blue) and emission (green) curves overlap partially —
this overlap region is precisely why a filter cube cannot simply split "excitation" from "emission"
at one sharp wavelength; it needs to make a deliberate trade-off about exactly where to draw that
line, which Section 3 simulates directly.

✏️ **Your turn.** Change `fwhm_nm` for `gfp_em` to `60` (a fluorophore with a much broader emission
spectrum). Does the excitation/emission overlap region grow or shrink? What does this suggest about
why some fluorophores are much easier to pair with a clean filter cube than others?

---
## 3 · The filter cube: excitation filter, dichroic mirror, emission filter

An epifluorescence filter cube has three optical elements working together along the same
light path: an **excitation filter** passes only a narrow band of wavelengths from a broadband lamp or
LED toward the specimen, tuned to the fluorophore's excitation peak; a **dichroic mirror**, angled at
45°, reflects that short-wavelength excitation light down onto the specimen while transmitting the
longer-wavelength emitted light back up toward the detector; and an **emission filter** blocks any
leftover excitation light and passes only the fluorophore's emission band to the camera.
`dichroic_transmission` models the dichroic's long-pass transmission curve, and the same Gaussian
`fluorophore_spectrum` shape doubles conveniently as a bandpass excitation-filter model.

In [ ]:
# ── 3.1 simulate a complete filter cube tuned to the GFP-like fluorophore ──
excitation_filter = fluorophore_spectrum(wavelengths_nm, peak_nm=488, fwhm_nm=15)  # narrow bandpass
dichroic_cutoff_nm = 505
dichroic = dichroic_transmission(wavelengths_nm, cutoff_nm=dichroic_cutoff_nm, steepness_nm=4.0)

def bandpass(wl, low_nm, high_nm, steepness_nm=4.0):
    return (dichroic_transmission(wl, low_nm, steepness_nm)
            * (1.0 - dichroic_transmission(wl, high_nm, steepness_nm)))

emission_filter = bandpass(wavelengths_nm, 500, 550)

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(wavelengths_nm, gfp_ex, color='blue', alpha=0.4, label='fluorophore excitation spectrum')
ax.plot(wavelengths_nm, gfp_em, color='green', alpha=0.4, label='fluorophore emission spectrum')
ax.plot(wavelengths_nm, excitation_filter, color='navy', ls='--', label='excitation filter transmission')
ax.plot(wavelengths_nm, dichroic, color='purple', ls=':', label='dichroic transmission (reflects below, transmits above)')
ax.plot(wavelengths_nm, emission_filter, color='darkgreen', ls='--', label='emission filter transmission')
ax.set_xlabel('wavelength (nm)'); ax.set_ylabel('transmission / intensity')
ax.set_title('A complete filter cube tuned to a GFP-like fluorophore')
ax.legend(fontsize=7, loc='upper left')
plt.tight_layout(); plt.show()

🔍 **What just happened?** The excitation filter (navy dashed) overlaps tightly with the
fluorophore's excitation spectrum (blue), so most excitation light actually reaches the fluorophore.
The dichroic (purple dotted) sits right in the gap between excitation and emission — reflecting
everything below its cutoff toward the specimen, transmitting everything above it toward the
detector. The emission filter (dark green dashed) then overlaps tightly with the emission spectrum
(green) while staying well clear of any leftover excitation light near 488 nm. This is a *well-chosen*
filter cube; Section 4 shows what happens when a second fluorophore's spectrum does not stay as
politely out of the way.

✏️ **Your turn.** Move `dichroic_cutoff_nm` down to `495`. Does the dichroic now clip off part of
the fluorophore's actual emission spectrum before it can reach the detector? Why would a real
instrument designer deliberately leave a small gap between the excitation peak and the dichroic
cutoff, rather than placing the cutoff as close to the excitation peak as possible?

---
## 4 · Spectral bleed-through: when two fluorophores' colors overlap

Add a second fluorophore to the same experiment, imaged through a second, independently-tuned
filter cube, and a new problem appears: each fluorophore's emission spectrum has some (however small)
intensity under the *other* fluorophore's emission filter. The fraction of a fluorophore's signal that
is detected in the wrong channel is called **spectral bleed-through** (or crosstalk), and it can be
computed directly by integrating each fluorophore's emission spectrum against each channel's
combined dichroic + emission-filter transmission curve.

In [ ]:
# ── 4.1 compute a 2x2 bleed-through matrix for a GFP-like/mCherry-like fluorophore pair ──
def detected_signal(emission_spectrum, channel_transmission):
    return np.trapezoid(emission_spectrum * channel_transmission, wavelengths_nm)

mcherry_ex = fluorophore_spectrum(wavelengths_nm, peak_nm=587, fwhm_nm=25)
mcherry_em = fluorophore_spectrum(wavelengths_nm, peak_nm=610, fwhm_nm=40)

green_channel = bandpass(wavelengths_nm, 500, 550)
red_channel = bandpass(wavelengths_nm, 600, 660)

fluorophores = {'GFP-like (em peak 509 nm)': gfp_em, 'mCherry-like (em peak 610 nm)': mcherry_em}
channels = {'green channel (500-550 nm)': green_channel, 'red channel (600-660 nm)': red_channel}

matrix = np.zeros((len(fluorophores), len(channels)))
for i, em in enumerate(fluorophores.values()):
    for j, ch in enumerate(channels.values()):
        matrix[i, j] = detected_signal(em, ch)

print(f"{'':32s}" + "".join(f"{name:>26s}" for name in channels))
for i, name in enumerate(fluorophores):
    print(f"{name:32s}" + "".join(f"{matrix[i, j]:>26.3f}" for j in range(len(channels))))

bleed_gfp_into_red = matrix[0, 1] / matrix[0, 0] * 100
bleed_mcherry_into_green = matrix[1, 0] / matrix[1, 1] * 100
print(f"\nGFP-like signal bleeding into the red channel: {bleed_gfp_into_red:.2f}% of its own-channel signal")
print(f"mCherry-like signal bleeding into the green channel: {bleed_mcherry_into_green:.2f}% of its own-channel signal")

🔍 **What just happened?** The diagonal of the matrix (each fluorophore detected in its *own*
intended channel) is much larger than the off-diagonal (each fluorophore detected in the *other*
channel) — but the off-diagonal values are not zero. That small but nonzero bleed-through percentage
is exactly the contamination this module's opening question warned about: it would be added directly
on top of any genuine colocalization signal, with no way for a downstream analysis to tell the two
apart from the image alone.

✏️ **Your turn.** Narrow the `red_channel` bandpass to `bandpass(wavelengths_nm, 620, 660)` (moving
its lower edge further from the GFP-like emission peak) and recompute `bleed_gfp_into_red`. Does
pushing the red channel's filter further from the green fluorophore's emission reduce bleed-through,
and at what cost to the red fluorophore's own detected signal?

### 📘 Lecture notes — bleed-through is a property of the fluorophore/filter PAIR, not either alone

**Definition.** Spectral bleed-through is the fraction of a fluorophore's emitted light detected
through a filter channel intended for a *different* fluorophore, computed here as the overlap
integral between one fluorophore's emission spectrum and another channel's transmission curve.

**Worked example.** The same GFP-like fluorophore that bleeds only slightly into a well-separated red
channel (Section 4) can bleed heavily into a poorly-chosen second channel — bleed-through is never a
fixed property of a single fluorophore, only of a specific fluorophore-and-filter-set combination,
as Section 5 demonstrates directly.

**One memorable point.** *A colocalization experiment's accuracy is decided, in part, weeks before
the microscope is ever turned on — at the moment the fluorophores and filter sets are chosen.*

---
## 5 · Choosing fluorophores and filters for multi-color imaging

Section 4's GFP-like/mCherry-like pair was deliberately well-separated in wavelength. Not every
convenient pair of fluorophores is: two fluorophores with closely-spaced emission peaks (for instance
a GFP-like and a YFP-like protein, both green-ish) inevitably bleed into each other far more heavily,
however carefully the filters are chosen, simply because their spectra themselves overlap more.

In [ ]:
# ── 5.1 compare a well-separated pair against a poorly-separated pair, using the same calculation ──
yfp_em = fluorophore_spectrum(wavelengths_nm, peak_nm=527, fwhm_nm=30)       # close to GFP-like
far_red_em = fluorophore_spectrum(wavelengths_nm, peak_nm=668, fwhm_nm=35)   # far from GFP-like

yfp_channel = bandpass(wavelengths_nm, 520, 560)
far_red_channel = bandpass(wavelengths_nm, 660, 720)

poor_pair_bleed = detected_signal(gfp_em, yfp_channel) / detected_signal(yfp_em, yfp_channel) * 100
good_pair_bleed = detected_signal(gfp_em, far_red_channel) / detected_signal(far_red_em, far_red_channel) * 100

print(f'GFP-like bleeding into a closely-spaced YFP-like channel:  {poor_pair_bleed:6.2f}%  (poorly-separated pair)')
print(f'GFP-like bleeding into a well-separated far-red channel:   {good_pair_bleed:6.2f}%  (well-separated pair)')

fig, ax = plt.subplots(figsize=(7, 3.3))
ax.fill_between(wavelengths_nm, gfp_em, color='green', alpha=0.3, label='GFP-like emission')
ax.fill_between(wavelengths_nm, yfp_em, color='gold', alpha=0.3, label='YFP-like emission (poor pairing)')
ax.fill_between(wavelengths_nm, far_red_em, color='darkred', alpha=0.3, label='far-red emission (good pairing)')
ax.set_xlabel('wavelength (nm)'); ax.set_ylabel('normalized intensity')
ax.set_title('Spectral separation decides how bad bleed-through can get')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The GFP-like/YFP-like pair's bleed-through percentage is dramatically
higher than the GFP-like/far-red pair's, purely because of how much their emission spectra overlap —
no choice of filter cube can fully fix this, because the fluorophores themselves are too spectrally
similar. This is exactly why experienced microscopists choose fluorophore combinations with large
spectral separation *before* worrying about which specific filter sets to buy.

✏️ **Your turn.** Try an intermediate case: a fluorophore with an emission peak at `560` nm,
imaged through a channel `bandpass(wavelengths_nm, 555, 600)`. Where does its bleed-through into the
original GFP-like green channel fall relative to the two cases computed above?

## 📌 Key takeaways

- The Jablonski diagram's absorb-relax-emit sequence makes the Stokes shift unavoidable: emitted
  light is always at a longer wavelength than absorbed light.
- A fluorophore's excitation and emission spectra, and a filter cube's excitation/dichroic/emission
  transmission curves, can all be modeled as simple parametric (Gaussian or logistic) curves.
- A filter cube's three elements work together to deliver excitation light to the specimen and only
  emission light to the detector, with the dichroic's cutoff sitting deliberately in the gap between
  the two spectra.
- Spectral bleed-through is the overlap integral between one fluorophore's emission and another
  channel's transmission curve — it depends on the fluorophore/filter pairing, not either alone.
- Choosing well-separated fluorophores is often a bigger lever against bleed-through than any
  subsequent choice of filter set.

## ✅ Checkpoint

**1.** A fluorophore's excitation peak is measured at 650 nm. Could its emission peak plausibly be at
630 nm?

<details><summary>Answer</summary>
No. The Stokes shift (Section 1) means emission always occurs at a longer wavelength (lower energy)
than excitation, as an unavoidable consequence of vibrational relaxation before emission. An emission
peak at a *shorter* wavelength than the excitation peak would violate this.
</details>

**2.** Two different research groups measure very different bleed-through percentages for "the same"
fluorophore. Is this necessarily a contradiction?

<details><summary>Answer</summary>
No. As Section 4's lecture note emphasizes, bleed-through is a property of the fluorophore-and-filter
PAIR, not the fluorophore alone. Two groups using different filter cubes (different dichroic cutoffs
or emission bandpass windows) can get genuinely different bleed-through numbers for the identical
fluorophore.
</details>

**3.** A new green fluorophore with an unusually broad emission spectrum is proposed as a replacement
for the GFP-like fluorophore in a two-color green/red experiment. Based on this module, what is the
single most important thing to check before adopting it?

<details><summary>Answer</summary>
How much its (broader) emission spectrum overlaps with the red channel's transmission curve —
exactly the `detected_signal`/bleed-through calculation from Section 4 — since Section 2's "Your turn"
exercise already showed that a broader emission spectrum tends to increase excitation/emission
overlap and, by the same logic, overlap with a neighboring channel.
</details>

## 🧠 Questions & discussion

- Spectral unmixing (a computational technique, distinct from the optical filtering in this module)
  uses the full emission spectra of all fluorophores in an experiment to mathematically separate
  their contributions, even with substantial filter overlap. Based on Section 4's bleed-through
  matrix, what information would such an algorithm need as input to do this?
- This module modeled excitation and emission spectra as simple, symmetric Gaussians. Real
  fluorophore spectra are often asymmetric, with a long tail toward longer wavelengths. Would this
  tend to make bleed-through estimates from the Gaussian model too optimistic or too pessimistic?
- A confocal microscope (a later module in this course) can use a tunable detector to select an
  arbitrarily narrow emission window, rather than a fixed physical emission filter. How might this
  change the bleed-through trade-offs described in this module?

## 🚀 Bonus — going further

- Add a third fluorophore and filter channel to Section 4's bleed-through matrix, and compute the
  full 3x3 crosstalk matrix for a three-color experiment.
- Model a more realistic, asymmetric emission spectrum (for instance, the sum of two Gaussians of
  different width on either side of the peak) and recompute a bleed-through percentage; compare it to
  the symmetric-Gaussian estimate.
- Sweep the red channel's lower bandpass edge continuously from 560 nm to 640 nm and plot
  `bleed_gfp_into_red` as a function of that edge position, to see the full trade-off curve implied
  by Section 4's "Your turn" exercise.

## 📝 Lab note

```text
Module: 07 — Fluorescence Microscopy Instrumentation
Date:
The fluorophore pair and filter sets I would choose for MY own multi-color experiment, and why:
A bleed-through percentage I would now want to measure before trusting a colocalization result:
```

## 📚 References

1. Lakowicz, J.R. (2006). *Principles of Fluorescence Spectroscopy* (3rd ed.). Springer.
2. Shaner, N.C., Steinbach, P.A., Tsien, R.Y. (2005). A guide to choosing fluorescent proteins.
   *Nature Methods*, 2(12), 905-909.
3. Waters, J.C. (2009). Accuracy and precision in quantitative fluorescence microscopy. *Journal of
   Cell Biology*, 185(7), 1135-1148.
4. Murphy, D.B., Davidson, M.W. (2012). *Fundamentals of Light Microscopy and Electronic Imaging*
   (2nd ed.). Wiley-Blackwell. — a fuller treatment of filter cube design and fluorophore spectra.

## ➡️ Next

Module 08 builds on this fluorescence foundation to cover the instrumentation that captures a
three-dimensional specimen rather than a single flat plane: confocal pinholes, multiphoton
excitation, TIRF's evanescent field, and single-molecule localization microscopy.